# Storage units
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/PyJobShop/PyJobShop/blob/main/examples/storage_units.ipynb)

> If you're using this notebook in Google Colab, be sure to install PyJobShop first by executing ```pip install pyjobshop``` in a cell.

This notebook demonstrates how to model storage units using renewable resources. In many real-world scheduling problems, a task places an item into limited storage, and a later task retrieves it, freeing the space. The key challenge is that the storage duration depends on the schedule itself.

We'll cover the following:

- A motivating example: scheduling vials and caps with limited cap storage
- The modeling trick: using `allow_idle=True` with synchronization constraints
- Scaling the storage capacity to allow multiple items in storage simultaneously

## Problem description

Consider a lab automation scenario where capped vials are processed by a robot arm. The processing steps for each vial are:

1. **Decap**: The robot arm and a decapper remove the cap from the vial.
2. **Use vial**: The robot arm processes the decapped vial (variable duration).
3. **Recap**: The robot arm and decapper put the cap back on.

While the vial is being used, the cap must be placed in a storage unit with limited capacity. If storage is full, no more vials can be decapped until a cap is retrieved. We model this storage as a renewable resource.

## Building the model

Let's start by creating the model with a robot arm, a decapper, and a cap storage with capacity 1.

In [ ]:
import random

from pyjobshop import Model

random.seed(42)

model = Model()

num_jobs = 3
jobs = [model.add_job() for _ in range(num_jobs)]

robot = model.add_machine(name="Robot")
decapper = model.add_machine(name="Decapper")
cap_storage = model.add_renewable(name="CapStorage", capacity=1)

For each job, we add three main tasks (decap, use vial, recap) with precedence constraints. The decap and recap tasks both require the robot and decapper, while using the vial only requires the robot.

Now for the key modeling trick: we need to track how long each cap occupies storage. Since this duration depends on the schedule, we can't set it upfront. Instead, we create a separate `CapInStorage` task for each job with `allow_idle=True`, which lets the task stretch beyond its nominal duration. We then pin the actual duration using synchronization constraints:

- `start_at_start(cap_in_storage, decap)` — storage starts when decapping starts
- `end_at_end(cap_in_storage, recap)` — storage ends when recapping ends

This task uses one unit of the renewable resource, so the capacity constraint prevents scheduling another decap while storage is full.

In [ ]:
for j, job in enumerate(jobs):
    task_decap = model.add_task(name=f"Decap{j}", job=job)
    task_usevial = model.add_task(name=f"UseVial{j}", job=job)
    task_recap = model.add_task(name=f"Recap{j}", job=job)

    model.add_mode(task_decap, [robot, decapper], duration=2)
    model.add_mode(task_usevial, robot, duration=random.randint(5, 20))
    model.add_mode(task_recap, [robot, decapper], duration=2)

    model.add_end_before_start(task_decap, task_usevial)
    model.add_end_before_start(task_usevial, task_recap)

    # Model the cap's time in storage as a separate task that holds
    # the renewable storage resource.
    task_cap_in_storage = model.add_task(
        name=f"CapInStorage{j}", job=job, allow_idle=True
    )
    model.add_mode(task_cap_in_storage, cap_storage, duration=0, demands=[1])

    model.add_start_at_start(task_cap_in_storage, task_decap)
    model.add_end_at_end(task_cap_in_storage, task_recap)

## Solving and visualizing

Let's solve the model and visualize the schedule.

In [ ]:
result = model.solve(display=False)
print(result)

In [ ]:
from pyjobshop.plot import plot_machine_gantt

plot_machine_gantt(result.best, model.data(), plot_labels=True)

The Gantt chart shows that only one cap is in storage at any time. Each `CapInStorage` task spans from the start of decapping to the end of recapping, preventing overlapping storage usage.

## Scaling storage capacity

The same pattern works with larger storage. Let's increase the capacity to 2, allowing two caps to be stored simultaneously. We also increase the number of jobs to 5 to better see the effect.

In [ ]:
random.seed(42)

model = Model()

num_jobs = 5
jobs = [model.add_job() for _ in range(num_jobs)]

robot = model.add_machine(name="Robot")
decapper = model.add_machine(name="Decapper")
cap_storage = model.add_renewable(name="CapStorage", capacity=2)

for j, job in enumerate(jobs):
    task_decap = model.add_task(name=f"Decap{j}", job=job)
    task_usevial = model.add_task(name=f"UseVial{j}", job=job)
    task_recap = model.add_task(name=f"Recap{j}", job=job)

    model.add_mode(task_decap, [robot, decapper], duration=2)
    model.add_mode(task_usevial, robot, duration=random.randint(5, 20))
    model.add_mode(task_recap, [robot, decapper], duration=2)

    model.add_end_before_start(task_decap, task_usevial)
    model.add_end_before_start(task_usevial, task_recap)

    task_cap_in_storage = model.add_task(
        name=f"CapInStorage{j}", job=job, allow_idle=True
    )
    model.add_mode(task_cap_in_storage, cap_storage, duration=0, demands=[1])

    model.add_start_at_start(task_cap_in_storage, task_decap)
    model.add_end_at_end(task_cap_in_storage, task_recap)

In [ ]:
result = model.solve(display=False)
print(result)

In [ ]:
plot_machine_gantt(result.best, model.data(), plot_labels=True)

We can also visualize the renewable resource usage to confirm that at most two caps are in storage at any given time:

In [ ]:
from pyjobshop.plot import plot_resource_usage

plot_resource_usage(result.best, model.data())

With capacity 2, the solver can overlap more operations, leading to a shorter makespan.

## Conclusion

This notebook showed how to model storage units in PyJobShop:

- Use a **renewable resource** to represent limited storage capacity.
- Create a separate task with `allow_idle=True` that holds the resource for the duration of storage.
- Pin the storage duration using `start_at_start` and `end_at_end` constraints.

This pattern generalizes to any situation where a resource is occupied for a variable, schedule-dependent duration.